# 07 — Linear Algebra
**Goal:** Use `np.linalg` for the matrix operations that underpin ML, statistics, and analytics.

```
dot / matmul  → matrix multiplication
linalg.solve  → solve Ax = b (linear systems)
linalg.inv    → matrix inverse
linalg.norm   → vector / matrix norms
linalg.eig    → eigenvalues & eigenvectors
linalg.svd    → singular value decomposition
```

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd

## 1. Dot product and matrix multiplication

In [ ]:
# Vector dot product — weighted sum
weights  = np.array([0.4, 0.3, 0.2, 0.1])   # channel mix weights
cvr      = np.array([3.0, 2.1, 4.5, 1.8])   # CVR per channel

blended_cvr = np.dot(weights, cvr)           # = Σ w_i * cvr_i
print('Weighted CVR:', blended_cvr.round(3))

# Verify manually
print('Manual:      ', (weights * cvr).sum().round(3))

In [ ]:
# Matrix multiplication: (m, k) @ (k, n) → (m, n)
# Apply 3 different weight sets (scenarios) to 4 channels
scenarios = np.array([
    [0.4, 0.3, 0.2, 0.1],   # current mix
    [0.5, 0.2, 0.2, 0.1],   # organic-heavy
    [0.2, 0.4, 0.2, 0.2],   # paid-heavy
])  # (3, 4)

cvr = np.array([3.0, 2.1, 4.5, 1.8])  # (4,)

# One matrix multiply instead of three dot products
blended = scenarios @ cvr   # (3,)
print('Blended CVR per scenario:', blended.round(3))

In [ ]:
# @ operator vs np.dot vs np.matmul — all equivalent for 2D
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print('A @ B     :\n', A @ B)
print('np.dot    :\n', np.dot(A, B))
print('np.matmul :\n', np.matmul(A, B))

## 2. Matrix inverse and solving linear systems

In [ ]:
# Solve Ax = b for x
# Attribution problem: 3 campaigns, 3 observed outcomes
# A[i,j] = contribution of campaign j in period i
A = np.array([
    [0.6, 0.3, 0.1],
    [0.2, 0.5, 0.3],
    [0.1, 0.2, 0.7],
])
b = np.array([2.8, 3.2, 4.1])   # observed CVR per period

# np.linalg.solve is more stable than computing inv(A) @ b
x = np.linalg.solve(A, b)
print('Base CVR per campaign:', x.round(3))

# Verify: A @ x should equal b
print('Residual (should be ~0):', np.round(A @ x - b, 10))

In [ ]:
# Matrix inverse — use only when you truly need the inverse itself
A = np.array([[2.0, 1.0], [5.0, 3.0]])
A_inv = np.linalg.inv(A)
print('A_inv:\n', A_inv)

# Verify: A @ A_inv ≈ I
print('A @ A_inv (≈ identity):\n', np.round(A @ A_inv, 10))

## 3. Norms — measuring distance and magnitude

In [ ]:
# L2 norm (Euclidean length) of a vector
v = np.array([3.0, 4.0])
print('L2 norm:', np.linalg.norm(v))          # 5.0 (3-4-5 triangle)

# L1 norm (Manhattan distance)
print('L1 norm:', np.linalg.norm(v, ord=1))   # 7.0

# Distance between two CVR vectors (channels over time)
cvr_jan = np.array([3.0, 2.1, 4.5, 1.8, 3.8])
cvr_feb = np.array([3.2, 2.0, 4.8, 1.9, 3.6])

diff = cvr_feb - cvr_jan
dist = np.linalg.norm(diff)
print(f'CVR shift Jan→Feb (L2 dist): {dist:.4f}')

## 4. Eigenvalues and eigenvectors

In [ ]:
# Eigendecomposition of a covariance matrix
# (basis of PCA — principal components are eigenvectors)

df = pd.read_csv('data/raw/funnel_data.csv')
cols = ['visita_landing', 'inicio_solicitud', 'datos_personales', 'activacion_tarjeta']
X = df[cols].to_numpy(dtype=float)

# Covariance matrix
cov = np.cov(X.T)   # np.cov expects (features, samples) → transpose
print('Cov shape:', cov.shape)

# Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eig(cov)

# Sort by descending eigenvalue
order = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

# Explained variance
explained = eigenvalues / eigenvalues.sum() * 100
for i, (ev, pct) in enumerate(zip(eigenvalues, explained)):
    print(f'  PC{i+1}: eigenvalue={ev:,.1f}  explains {pct:.1f}%')

## 5. Least-squares regression — OLS with NumPy

In [ ]:
# Fit: activations = β0 + β1*sessions using OLS (closed form)
df = pd.read_csv('data/raw/funnel_data.csv')

X_raw = df['visita_landing'].to_numpy(dtype=float)
y     = df['activacion_tarjeta'].to_numpy(dtype=float)

# Design matrix with intercept column
X = np.column_stack([np.ones(len(X_raw)), X_raw])  # (n, 2)

# np.linalg.lstsq solves min ||Xβ - y||² — numerically stable
beta, residuals, rank, sv = np.linalg.lstsq(X, y, rcond=None)
print(f'Intercept (β0): {beta[0]:.4f}')
print(f'Slope     (β1): {beta[1]:.4f}')

# Predictions
y_hat = X @ beta
ss_res = np.sum((y - y_hat) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
print(f'R²: {r2:.4f}')

## Summary
| Operation | NumPy |
|---|---|
| Dot product | `np.dot(a, b)` or `a @ b` |
| Matrix multiply | `A @ B` |
| Solve Ax = b | `np.linalg.solve(A, b)` |
| Matrix inverse | `np.linalg.inv(A)` |
| L2 norm | `np.linalg.norm(v)` |
| Eigendecomposition | `np.linalg.eig(A)` |
| Least squares | `np.linalg.lstsq(X, y)` |
| Covariance matrix | `np.cov(X.T)` |

**Next:** `08_real_world.ipynb` — an end-to-end analytics pipeline using only NumPy.